# 第13讲｜从建校差异到教育回报

继续同一INPRES项目。目标：共同工资样本 → 工具条件 → 第一阶段与简约式 → 2SLS → 弱工具与AR集合 → 研究备忘录。

不是以得到显著正系数为成功标准。当前简化规格的工具很弱，要保留这个结果。

## 准备
使用README中的相同环境与目录。即使刚运行完01，也从本Notebook第一格重新开始，避免隐藏状态。

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from inpres_project import (load_data, prepare, main_sample, four_cells, did,
                            fit_wls, estimate, ar_confidence, SHA256)
OUT = Path("results")
OUT.mkdir(exist_ok=True)
plt.rcParams.update({"figure.figsize": (9, 4.8), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
TEAL, AMBER = "#197f7a", "#b87921"
def save_json(name, value):
    (OUT / name).write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                      allow_nan=False) + "\n", encoding="utf-8")


## P7｜共同工资样本（课件第6页）
工资缺失不补0；`lhwage`已是自然对数，不再次取对数。所有阶段使用相同观察、权重和外部控制。

In [ ]:
df = prepare(load_data())
w = main_sample(df, wage=True)
assert len(w) == 31061 and w.district.nunique() == 290
assert w[["yeduc", "lhwage", "high", "young", "z", "weight"]].notna().all().all()
sample_audit = {"n": len(w), "districts": int(w.district.nunique()),
                "weight": "survey observation weight", "controls": ["high", "young"],
                "endogenous": "yeduc", "instrument": "z=high*young"}
pd.DataFrame([sample_audit]).to_csv(OUT / "sample_iv.csv", index=False)
display(pd.Series(sample_audit))

## P8｜候选工具的条件（课件第14页）
先写`iv_assumptions.md`：相关性准备计算什么；外生比较根据什么；哪条Z→Y直接通路可能破坏排除限制。
教育年限是多值变量，不能机械把IV斜率说成所有人的ATE或简单二元处理LATE。

### 同样本OLS基线
保留关联斜率，不预先给它因果名称。

In [ ]:
ols_fit = fit_wls("lhwage ~ high + young + yeduc", w)
ols_result = estimate(ols_fit, "yeduc", w)
display(pd.DataFrame([ols_result]))

## P9｜第一阶段、简约式与比率（课件第22页）
π的单位是教育年，ρ的单位是log工资。先核查共同样本，不把01中的教育主样本差分当分母。
比率点估计有代数对应，不代表可以把两个区间端点相除。

In [ ]:
first = fit_wls("yeduc ~ high + young + z", w)
reduced = fit_wls("lhwage ~ high + young + z", w)
pi, rho = first.params["z"], reduced.params["z"]
wald = rho / pi
stages = {"first": estimate(first, "z", w),
          "reduced": estimate(reduced, "z", w), "ratio": float(wald)}
save_json("stages.json", stages)
display(pd.DataFrame([{"equation": "first stage", **stages["first"]},
                      {"equation": "reduced form", **stages["reduced"]}]))
print(f"rho / pi = {wald:.6f}")

## P10｜正式2SLS与算法核对（课件第29页）
两次WLS可以理解点估计，第二阶段普通SE不能替代IV方差。
下列公式外部是外生控制，方括号内是内生变量与排除工具。
显示区间显式使用t(290−1)，不是不加说明地复制软件默认区间。

In [ ]:
from linearmodels.iv import IV2SLS
iv = IV2SLS.from_formula(
    "lhwage ~ 1 + high + young + [yeduc ~ z]", w,
    weights=w["weight"]).fit(cov_type="clustered", clusters=w["district"], debiased=True)
iv_result = estimate(iv, "yeduc", w)
assert abs(iv_result["beta"] - wald) < 1e-8
diagnostics = iv.first_stage.diagnostics
display(diagnostics)
display(pd.DataFrame([{"model": "OLS", **ols_result}, {"model": "2SLS", **iv_result}]))
save_json("iv_results.json", {"sample": sample_audit, "ols": ols_result, "iv": iv_result,
                             "diagnostics": diagnostics.reset_index().to_dict("records")})

### 点与常规区间一起保留
本图只是常规区间；当前第一阶段弱，不能替代接下来的AR集合。两种估计器大小不同，不证明某个更接近真值。

In [ ]:
fig, ax = plt.subplots()
for y, label, model, color in [(1, "OLS association", ols_result, TEAL),
                              (0, "2SLS conventional CI", iv_result, AMBER)]:
    b = model["beta"]
    ax.errorbar(b, y, xerr=[[b-model["ci"][0]], [model["ci"][1]-b]],
                fmt="o", color=color, capsize=5)
ax.axvline(0, color="gray", linewidth=1)
ax.set(yticks=[0, 1], yticklabels=["2SLS", "OLS"], ylim=(-.7, 1.7),
       xlabel="Log hourly wage per year of education",
       title="Same wage sample: conventional 95% district-cluster intervals")
fig.tight_layout()
fig.savefig(OUT / "iv_comparison.png", dpi=160)
plt.show()

## P11｜弱工具与AR检验反演（课件第36页）
不先除以π。对候选β₀回归Y−β₀S，检验z还能否解释剩余结果。
单工具线性模型可以用联合聚类协方差解二次不等式；函数保留无限端，不用有限网格边界冒称区间。
这仍依赖工具有效性；渐近聚类AR不是排除限制的检验。

In [ ]:
ar = ar_confidence(w, first, reduced)
assert ar["kind"] == "all-real"
display(pd.Series(ar))
save_json("ar_confidence.json", ar)
# 用直接回归独立核查若干候选值，并非靠这几个值认定整个集合。
candidate_checks = []
for beta0 in [-10.0, -1.0, 0.0, float(wald), 1.0, 10.0]:
    candidate = w.assign(ar_outcome=w.lhwage.astype(float)-beta0*w.yeduc.astype(float))
    ar_fit = fit_wls("ar_outcome ~ high + young + z", candidate)
    t2 = float(ar_fit.tvalues["z"]**2)
    expected = (rho-beta0*pi)**2 / (
        reduced.bse["z"]**2 + beta0**2*first.bse["z"]**2
        - 2*beta0*ar["covarianceRF_FS"])
    assert np.isclose(t2, expected, rtol=1e-5, atol=1e-7)
    candidate_checks.append({"beta0": beta0, "cluster_t2": t2,
                             "accepted_95": bool(t2 <= ar["critical2"])})
display(pd.DataFrame(candidate_checks))

### 解释本项目的实际结果
AR的95%集合为全体实数：这条简化工具没有给出有限的信息性教育回报范围。
不是证明效应为零，也不是程序运行失败。

### 权重敏感性
不加权结果作为预先保留的比较，不据结果选择权重。

In [ ]:
iv_u = IV2SLS.from_formula("lhwage ~ 1 + high + young + [yeduc ~ z]", w).fit(
    cov_type="clustered", clusters=w.district, debiased=True)
unweighted_iv = estimate(iv_u, "yeduc", w)
save_json("iv_sensitivity.json", {"weighted": iv_result, "unweighted": unweighted_iv})
display(pd.DataFrame([{"weights": "survey", **iv_result},
                      {"weights": "none", **unweighted_iv}]))

## P12｜研究备忘录（课件第43页）
写`results/memo_iv.md`，并把两讲整合为`memo.md`：

1. 地区×队列DID：教育主样本、结果、估计对象与条件。
2. 工资样本：第一阶段、简约式、2SLS，不能混用全体教育样本分母。
3. 第一阶段弱，AR集合无有限信息边界；并列讨论排除限制与工资样本选择。
4. 一项具体后续资料需求，而不是自动搜索显著规格。

## 重跑与交接
重启内核从头运行；核对文件、n、权重、控制、聚类与区间类型。保存自己的第一版，不用参考答案覆盖。

来源：[Duflo (2001)](https://www.aeaweb.org/articles?id=10.1257/aer.91.4.795)；
[Roodman复核](https://arxiv.org/abs/2207.09036)；
[linearmodels公式与推断接口](https://bashtage.github.io/linearmodels/iv/examples/basic-examples.html)。
本模型不复制原论文全部规格，也不因为工具来自政策就视为随机实验。